# Comparación visual FIgLib: 224×224 vs 384×384

Este notebook reproduce el preprocesado de evaluación usado en los modelos **Full224** y **Full384**:

- Full224: `Resize(256) -> CenterCrop(224)`
- Full384: `Resize(440) -> CenterCrop(384)`

Además, guarda las dos imágenes procesadas y una figura comparativa lista para usar en la memoria.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Selecciona una imagen del dataset

Cambia únicamente `IMAGE_PATH`. Conviene elegir una imagen donde el humo sea **pequeño o lejano**.


In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms

# CAMBIA ESTA RUTA POR LA IMAGEN QUE QUIERAS USAR
IMAGE_PATH = "/content/drive/MyDrive/TFM_UCM/04_Datos/TFM_Incendios/02_data/raw/figlib/full_sequences_full_511/20180706_West_lp-n-mobo-c/20180706_West_lp-n-mobo-c/1530903841_+02160.jpg"

image_path = Path(IMAGE_PATH)
assert image_path.exists(), f'No existe la imagen: {image_path}'

img = Image.open(image_path).convert('RGB')
print('Imagen:', image_path.name)
print('Resolución original:', img.size)

plt.figure(figsize=(12, 7))
plt.imshow(img)
plt.title(f'Imagen original — {img.size[0]}×{img.size[1]}')
plt.axis('off')
plt.show()


## 2. Aplicar exactamente el preprocesado de evaluación


In [ ]:
transform_224 = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
])

transform_384 = transforms.Compose([
    transforms.Resize(440),
    transforms.CenterCrop(384),
])

img_224 = transform_224(img)
img_384 = transform_384(img)

print('Full224:', img_224.size)
print('Full384:', img_384.size)


## 3. Comparación lado a lado


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(img_224)
axes[0].set_title('Full224 — Resize(256) + CenterCrop(224)')
axes[0].axis('off')

axes[1].imshow(img_384)
axes[1].set_title('Full384 — Resize(440) + CenterCrop(384)')
axes[1].axis('off')

fig.suptitle('Comparación del preprocesado de entrada', fontsize=14)
plt.tight_layout()
plt.show()


## 4. Guardar imágenes y figura comparativa


In [ ]:
OUTPUT_DIR = Path('/content/drive/MyDrive/figlib_comparacion_resolucion')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

img_224.save(OUTPUT_DIR / 'imagen_224.jpg', quality=95)
img_384.save(OUTPUT_DIR / 'imagen_384.jpg', quality=95)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_224)
axes[0].set_title('Entrada 224×224')
axes[0].axis('off')
axes[1].imshow(img_384)
axes[1].set_title('Entrada 384×384')
axes[1].axis('off')
plt.tight_layout()

comparison_path = OUTPUT_DIR / 'comparacion_224_384.png'
fig.savefig(comparison_path, dpi=300, bbox_inches='tight')
plt.show()

print('Guardado en:', OUTPUT_DIR)
print('-', OUTPUT_DIR / 'imagen_224.jpg')
print('-', OUTPUT_DIR / 'imagen_384.jpg')
print('-', comparison_path)


## 5. Zoom opcional de la zona de humo

Introduce coordenadas sobre la imagen original como `(x_izq, y_sup, x_der, y_inf)`.


In [ ]:
ROI = None  # Ejemplo: (500, 300, 850, 600)

if ROI is not None:
    roi = img.crop(ROI)
    roi_224 = transforms.Resize((224, 224))(roi)
    roi_384 = transforms.Resize((384, 384))(roi)

    fig_zoom, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(roi_224)
    axes[0].set_title('Misma región a 224×224')
    axes[0].axis('off')
    axes[1].imshow(roi_384)
    axes[1].set_title('Misma región a 384×384')
    axes[1].axis('off')
    plt.tight_layout()

    zoom_path = OUTPUT_DIR / 'comparacion_zoom_224_384.png'
    fig_zoom.savefig(zoom_path, dpi=300, bbox_inches='tight')
    plt.show()
    print('Zoom guardado en:', zoom_path)
else:
    print('ROI=None. Define una ROI si quieres generar el zoom comparativo.')


### Nota metodológica

La figura ilustra la diferencia en información espacial disponible para el modelo. No debe interpretarse como una demostración visual por sí sola de que 384 sea superior; esa conclusión se apoya en la comparación experimental y el bootstrap pareado por evento.
